# Make channel names for human images

Run with `mesmer_pypi` conda environment

In [1]:
import numpy as np
import tifffile
import os 
import argparse
import re
import pandas as pd
from xml.etree import ElementTree
from typing import List, Union
from pathlib import Path 

In [2]:
def get_markers_from_img(img_file: Union[str, Path]) -> pd.DataFrame:
    with tifffile.TiffFile(img_file) as tif:
        marker_list = []
        for page in tif.series[0].pages:
            xml_description = ElementTree.fromstring(page.description)
            marker_list.append(xml_description.find('Biomarker').text)

    markers_df = pd.DataFrame({
        "cycle"       : [1] * len(marker_list),
        "marker_name" : marker_list
    })
    markers_df = markers_df.reset_index().rename(columns={'index': 'row_num'})
    markers_df = markers_df[["cycle", "marker_name", "row_num"]]
    return markers_df

In [3]:
hum_block_dir = "/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/"
block_qptiffs = [os.path.join(hum_block_dir, img_file) for img_file in os.listdir(hum_block_dir) if re.search("qptiff", img_file)]
block_qptiffs

['/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu_aging_block5_batch5-3_20250329.qptiff',
 '/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu_aging_block6_batch5-4_20250331.qptiff',
 '/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu-aging_block4_batch5-2_20250328.qptiff',
 '/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu_block2_batch4-4_20241217_9-2_Scan1.qptiff',
 '/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu_aging_block3-1_batch5-1_20250327_9-11.qptiff',
 '/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/Hu_block1_2nd_batch4-3_2_20241220_9-2_Scan1.qptiff']

In [14]:
for block_qptiff in block_qptiffs: 
    marker_df = get_markers_from_img(block_qptiff)
    marker_df = marker_df.loc[(~marker_df["marker_name"].str.contains(" rinse")) &
                              (~marker_df["marker_name"].isin(['Aire'])), :].copy()
    print(marker_df.head(n=20))
    marker_df.to_csv(re.sub(".qptiff", ".csv", block_qptiff), index=False)

    cycle      marker_name  row_num
0       1             DAPI        0
1       1       Podoplanin        1
2       1               K5        2
3       1             CD68        3
4       1             CD44        4
5       1             CD31        5
6       1              CD4        6
7       1              K14        7
8       1  Pan-Cytokeratin        8
9       1            CD11c        9
10      1            HLA-A       10
11      1         Vimentin       11
12      1             CD40       12
13      1              CD8       13
14      1             CD20       14
15      1             CD3e       15
16      1               K8       16
18      1           HLA-DR       18
19      1             Ki67       19
20      1              K10       20
    cycle      marker_name  row_num
0       1             DAPI        0
1       1       Podoplanin        1
2       1               K5        2
3       1             CD68        3
4       1             CD44        4
5       1             CD31  